[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/125/blob/main/07_dataframes/07_3_Spotting_Problems.ipynb)

# 07.3: Spotting Problems in Data

So far we have mostly taken the data at face value. Real data has gaps and errors, and pandas will calculate an answer from it anyway, without any warning. This short notebook shows how to check a dataset for common problems and how to handle the simple ones. A full treatment of data cleaning is a topic for a later course. The goal here is to get in the habit of checking before you trust a result.

## The data

This data comes from the Centers for Disease Control and has one row for each reported COVID-19 case in McLean County, Illinois, from March 2020 to March 2021. We will keep a few of the columns.

In [ ]:
import pandas as pd

covid = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_covid_mclean_county_retrieved_4_1_21.csv")
covid = covid[["case_month", "age_group", "sex", "race", "hosp_yn"]]
covid.head()

## Finding missing values

`.isna().sum()` on a DataFrame counts the missing values in each column. This is the same `.isna()` you saw in 06.3 for a series.

In [ ]:
print(len(covid))
covid.isna().sum()

There are 15334 rows. `age_group` has 134 missing values, `sex` has 679 and `race` has 1829. `case_month` and `hosp_yn` appear to be complete.

## Missing data does not always look missing

`.value_counts()` skips missing values unless you pass `dropna=False`, which also gives you a look at every value in a column.

In [ ]:
covid["sex"].value_counts(dropna=False)

Unknown sex is recorded in three different ways: 679 are `NaN`, 186 say `Unknown` and 44 say `Missing`. pandas counts only the `NaN` values as missing. The other two are ordinary strings to it. The problem is much larger in `hosp_yn`, which records whether the person was hospitalized.

In [ ]:
covid["hosp_yn"].value_counts(dropna=False)

`.isna().sum()` reported nothing wrong with this column, yet 10789 rows say `Missing` and 1109 say `Unknown`. Only 3436 of the 15334 rows have a real answer. If you computed the percentage of cases that were hospitalized using all the rows, you would get 424 out of 15334, or about 3 percent. Using only the rows with an answer gives 424 out of 3436, or about 12 percent. So look at the values in a column, and do not rely on `.isna()` alone.

## What to do about it: dropping and filling

`.dropna()` removes rows with missing values. By default it drops a row if **any** column is missing, which can lose far more than you intended. The `subset=` argument limits it to the columns you name.

In [ ]:
print(len(covid.dropna()))
print(len(covid.dropna(subset=["age_group"])))

Dropping on every column leaves 13505 rows, while dropping only where `age_group` is missing leaves 15200, which loses just the 134 rows we expected. `.fillna()` replaces missing values with one you choose. That can be a good way to keep a category of unknowns visible in your counts. Both of these return a new result, and `covid` does not change unless you assign it back.

In [ ]:
covid["age_group"].fillna("Unknown").value_counts()

Rows that say `Missing` or `Unknown` are ordinary strings, so you remove them with a mask, the way you learned in 07.2.

In [ ]:
answered = covid[covid["hosp_yn"].isin(["Yes", "No"])]
print(len(answered))
(answered["hosp_yn"] == "Yes").mean()

This confirms the 3436 rows and a share of about 0.123. Whichever choice you make, it is good practice to report how many rows you left out, since the people with an answer may differ from the people without one.

## Values that cannot be right

Some problems have nothing to do with missing values. In 07.1, the Broadway `Capacity` column had a max of 892. A mask makes it easy to see how many rows look wrong.

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_corgis_broadway.csv")
print((df["Capacity"] > 100).sum())
print((df["Gross Potential"] == 0).sum())
df["Capacity"].sort_values(ascending=False).head(3)

`Capacity` is above 100 in 2129 rows. Nearly all of those, 2117 of them, are at 110 or below, which is believable because a theatre can sell some standing room. The other 12 go as high as 892, which is not believable. `Gross Potential` is 0 in 2129 rows as well (the matching count is a coincidence, since they are mostly different rows). Those zeros are mostly from the mid-1990s and appear alongside real sales, so a 0 here most likely means the value was not recorded, even though it looks like a real number. Nothing in the data tells you this directly. You find it by comparing the numbers with what you know about the world.

## Numbers stored as text

If one value in a column is not a number, the whole column is treated as text, and math on it will fail. `pd.to_numeric()` converts a series to numbers. With `errors="coerce"`, anything it cannot convert becomes `NaN`, and then the statistics you already know will skip it.

In [ ]:
ages = pd.Series(["34", "27", "Unknown", "51"])
pd.to_numeric(ages, errors="coerce")

`34`, `27` and `51` are now numbers, and `Unknown` has become `NaN`.

## What's next

You can now check a dataset for gaps and impossible values before you use it. Once you trust the data, the next question is usually a comparison, such as whether musicals earn more than plays. Notebook 07.4 shows how to compare groups.